In [2]:
from pyspark.sql import *

import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = SparkSession.builder \
    .appName("spark_lab_4") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/24 11:55:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/24 11:55:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/24 11:55:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


# JOIN 
- pyspark join is used to combine two dataframes.
- pyspark join operation combines data from two or more datasets based on a common colums or key.It is a fundamental operation in pyspark and is similar to SQL Joins.

#### It supports all basic join type operations availbale in traditional SQL like INNER, LEFT OUTER,RIGHT OUTER,LEFT ANTI,LEFT SEMI,CROSS,SLEF JOIN

- Innter Join     : Returns only the row with matching keys in both dataframes.
- Left Join       : Returns all rows from the left dataframe and matching row from the right dataframe
- Right Join      : Returns all rows from the right dataframe and matching row from the left dataframe
- Full Join       : Returns all rows from both datafrmes ,including matching and non-matching rows.
- Left Semi Join  : Returns all rows from the left dataframe where there is a match in the right dataframe
- Left Anti Join  : Returns all rows from the left dataframe where there is a no match in the right dataframe

In [11]:
emp_df = spark.read.csv("datasets/emp_2.csv", header=True, inferSchema=True)
dept_df = spark.read.csv("datasets/department.csv", header=True, inferSchema=True) 
emp_df.show()
dept_df.show()

+------+------+------+-----+
|emp_id|  name|salary|  loc|
+------+------+------+-----+
|     1|manish| 20000|india|
|     2|  hari|  5000|   UK|
|     3| rahul| 10000|india|
|     4|   nil| 20000|india|
|     5|   sil| 24000|   UK|
|     6|  neha| 17000|   UK|
+------+------+------+-----+

+----+----------+------------+
|user|department| designation|
+----+----------+------------+
|   1|        it|   software |
|   2|     sales|data analyst|
|   3|     other|     trainer|
|   4|        it|data analyst|
|   5|     sales|   marketing|
|   6|        it|data analyst|
+----+----------+------------+



In [13]:
emp_df.join(dept_df, emp_df.emp_id == dept_df.user, 'inner').show()

+------+------+------+-----+----+----------+------------+
|emp_id|  name|salary|  loc|user|department| designation|
+------+------+------+-----+----+----------+------------+
|     1|manish| 20000|india|   1|        it|   software |
|     2|  hari|  5000|   UK|   2|     sales|data analyst|
|     3| rahul| 10000|india|   3|     other|     trainer|
|     4|   nil| 20000|india|   4|        it|data analyst|
|     5|   sil| 24000|   UK|   5|     sales|   marketing|
|     6|  neha| 17000|   UK|   6|        it|data analyst|
+------+------+------+-----+----+----------+------------+



In [14]:
emp_df.join(dept_df,emp_df.emp_id == dept_df.user, 'left').show()

+------+------+------+-----+----+----------+------------+
|emp_id|  name|salary|  loc|user|department| designation|
+------+------+------+-----+----+----------+------------+
|     1|manish| 20000|india|   1|        it|   software |
|     2|  hari|  5000|   UK|   2|     sales|data analyst|
|     3| rahul| 10000|india|   3|     other|     trainer|
|     4|   nil| 20000|india|   4|        it|data analyst|
|     5|   sil| 24000|   UK|   5|     sales|   marketing|
|     6|  neha| 17000|   UK|   6|        it|data analyst|
+------+------+------+-----+----+----------+------------+



### Union and Union all
- union() and unionAll() transformations are used to merge two or more DataFrame's of the same schema or structure

In [30]:

data1 = [
        ("james","sales","NY",30000),
        ("Ram","software","Mumbai",90000),
        ("Arjun","Hr","pune",30000),
        ("sara","sales","Geramny",40000),
        ("Amruta","sales","Kerla",43000)
         ]
columns = ["emp_name","department","location","salary"]

df1 = spark.createDataFrame(data=data1,schema=columns)
df1.show()


data2 = [
        ("Rahul","sales","Delhi",34000),
        ("John","software","Cananda",120000),
        ("Natasha","Hr","Usa",23000),
        ("Amruta","sales","Kerla",43000)
         ]
df2 =spark.createDataFrame(data=data2,schema=columns)
df2.show()

+--------+----------+--------+------+
|emp_name|department|location|salary|
+--------+----------+--------+------+
|   james|     sales|      NY| 30000|
|     Ram|  software|  Mumbai| 90000|
|   Arjun|        Hr|    pune| 30000|
|    sara|     sales| Geramny| 40000|
|  Amruta|     sales|   Kerla| 43000|
+--------+----------+--------+------+

+--------+----------+--------+------+
|emp_name|department|location|salary|
+--------+----------+--------+------+
|   Rahul|     sales|   Delhi| 34000|
|    John|  software| Cananda|120000|
| Natasha|        Hr|     Usa| 23000|
|  Amruta|     sales|   Kerla| 43000|
+--------+----------+--------+------+



In [31]:
df1.printSchema()

root
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)
 |-- salary: long (nullable = true)



In [32]:
df2.printSchema()

root
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)
 |-- salary: long (nullable = true)



In [33]:
df1.union(df2).show()

+--------+----------+--------+------+
|emp_name|department|location|salary|
+--------+----------+--------+------+
|   james|     sales|      NY| 30000|
|     Ram|  software|  Mumbai| 90000|
|   Arjun|        Hr|    pune| 30000|
|    sara|     sales| Geramny| 40000|
|  Amruta|     sales|   Kerla| 43000|
|   Rahul|     sales|   Delhi| 34000|
|    John|  software| Cananda|120000|
| Natasha|        Hr|     Usa| 23000|
|  Amruta|     sales|   Kerla| 43000|
+--------+----------+--------+------+



In [34]:
df1.unionAll(df2).show()   #unionAll is depricated now

+--------+----------+--------+------+
|emp_name|department|location|salary|
+--------+----------+--------+------+
|   james|     sales|      NY| 30000|
|     Ram|  software|  Mumbai| 90000|
|   Arjun|        Hr|    pune| 30000|
|    sara|     sales| Geramny| 40000|
|  Amruta|     sales|   Kerla| 43000|
|   Rahul|     sales|   Delhi| 34000|
|    John|  software| Cananda|120000|
| Natasha|        Hr|     Usa| 23000|
|  Amruta|     sales|   Kerla| 43000|
+--------+----------+--------+------+



In [35]:
#union with unique record
df1.union(df2).distinct().show()

+--------+----------+--------+------+
|emp_name|department|location|salary|
+--------+----------+--------+------+
|   james|     sales|      NY| 30000|
|     Ram|  software|  Mumbai| 90000|
|   Arjun|        Hr|    pune| 30000|
|    sara|     sales| Geramny| 40000|
|  Amruta|     sales|   Kerla| 43000|
|   Rahul|     sales|   Delhi| 34000|
|    John|  software| Cananda|120000|
| Natasha|        Hr|     Usa| 23000|
+--------+----------+--------+------+



# fill and fillna
- In pyspark , fillna() from dataFrame class or fill() from DataFrameNa Functions is used to replace Null/None values on all or selected multiple columns with either zero(0), empty string, space, or any constant literal values

In [47]:
df = spark.read.csv("datasets/sample.csv", header=True,)
df.show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|               NULL|   PR|     30100|
|  2|    704|    NULL|PASEO COSTA DEL SUR|   PR|      NULL|
|  3|    709|    NULL|       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|               NULL|   TX|      NULL|
+---+-------+--------+-------------------+-----+----------+



In [48]:
#fill blank value where null values are present
df.na.fill("unknown").show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|            unknown|   PR|     30100|
|  2|    704| unknown|PASEO COSTA DEL SUR|   PR|   unknown|
|  3|    709| unknown|       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|            unknown|   TX|   unknown|
+---+-------+--------+-------------------+-----+----------+



In [49]:
df.fillna("").show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|                   |   PR|     30100|
|  2|    704|        |PASEO COSTA DEL SUR|   PR|          |
|  3|    709|        |       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|                   |   TX|          |
+---+-------+--------+-------------------+-----+----------+



In [50]:
#replace in specif colums
df.na.fill("",["type"]).show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|               NULL|   PR|     30100|
|  2|    704|        |PASEO COSTA DEL SUR|   PR|      NULL|
|  3|    709|        |       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|               NULL|   TX|      NULL|
+---+-------+--------+-------------------+-----+----------+



In [51]:
df.na.fill("",["type"]).na.fill("unknown",["city"]).show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|            unknown|   PR|     30100|
|  2|    704|        |PASEO COSTA DEL SUR|   PR|      NULL|
|  3|    709|        |       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|            unknown|   TX|      NULL|
+---+-------+--------+-------------------+-----+----------+



In [52]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- population: string (nullable = true)



In [55]:
#lets enable inferSchema
df = spark.read.csv("datasets/sample.csv",header=True,inferSchema=True)
df.printSchema()
#here we can see population datatype in integer

root
 |-- id: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- population: integer (nullable = true)



In [56]:
df.show()

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|               NULL|   PR|     30100|
|  2|    704|    NULL|PASEO COSTA DEL SUR|   PR|      NULL|
|  3|    709|    NULL|       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|               NULL|   TX|      NULL|
+---+-------+--------+-------------------+-----+----------+



In [60]:
df.na.fill(0).show() # we can see here all null values from population columns are conveting 0

+---+-------+--------+-------------------+-----+----------+
| id|zipcode|    type|               city|state|population|
+---+-------+--------+-------------------+-----+----------+
|  1|    704|STANDARD|               NULL|   PR|     30100|
|  2|    704|    NULL|PASEO COSTA DEL SUR|   PR|         0|
|  3|    709|    NULL|       BDA SAN LUIS|   PR|      3700|
|  4|  76166|  UNIQUE|  CINGULAR WIRELESS|   TX|     84000|
|  5|  76177|STANDARD|               NULL|   TX|         0|
+---+-------+--------+-------------------+-----+----------+

